In [0]:
import pyspark.sql.functions as F


In [0]:
silver_table = spark.read.table("databricks_practice.silver.silver_table")
dim_sku = spark.read.table("databricks_practice.silver.dim_sku")
dim_tamanho = spark.read.table("databricks_practice.silver.dim_tamanho")
dim_mercado = spark.read.table("databricks_practice.silver.dim_mercado")
dim_produto_scd2 = spark.read.table("databricks_practice.silver.dim_product_scd2")


In [0]:
# # Procurando a chave da tabela dim_tamanho
# display(
#     silver_table.groupby("nike_size", "product_id", "country_code")
#     .count()
#     .filter(F.col("count") > 1)
# )

In [0]:
fact_table = silver_table.alias("s") \
    .join(
        dim_sku.alias("sku"),
        (F.col("s.sku") == F.col("sku.sku")) &
        (F.col("s.product_id") == F.col("sku.product_id")),
        "inner"
    ) \
    .join(
        dim_mercado.alias("merc"),
        F.col("s.country_code") == F.col("merc.country_code"),
        "inner"
    ) \
    .join(
        dim_produto_scd2.alias("prod"),
        (
        (F.col("s.product_id") == F.col("prod.product_id")) &
        (F.col("s.snapshot_date") >= F.col("prod.start_dt")) &
        (F.col("s.snapshot_date") < F.col("prod.end_dt"))
    ),
        "inner"
    ) \
    .select(
        F.col("s.*"),
        F.col("sku.sku_sk"),
        F.col("merc.location_sk"),
        F.col("prod.product_sk"), F.col("current_flag")
    )

In [0]:
print("sku:", dim_sku.count())
print("merc:", dim_mercado.count())
print("prod:", dim_produto_scd2.count())

In [0]:
display(fact_table)

In [0]:
fact_table.write \
    .mode("overwrite") \
    .saveAsTable("databricks_practice.silver.fact_table")

In [0]:
%sql
SELECT * FROM databricks_practice.silver.fact_table WHERE current_flag = "N"